In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/music-genre-capstone

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

Mounted at /content/drive
[Errno 2] No such file or directory: '/content/music-genre-capstone'
/content
PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128


In [2]:
import os

PROJECT = "/content/drive/MyDrive/music-genre-capstone"
CHECKPOINT = PROJECT + "/models/transfer_learning/best_resnet18_finetuned_extended.pth"

print("Project exists:", os.path.exists(PROJECT))
print("Checkpoint exists:", os.path.exists(CHECKPOINT))

print("\nProject contents:")
print(os.listdir(PROJECT))

Project exists: True
Checkpoint exists: True

Project contents:
['data', 'features', 'checkpoints', 'test_1gb.bin', 'models', 'results']


In [3]:
import torch

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in checkpoint.keys():
        print(" -", key)

Checkpoint type: <class 'dict'>

Checkpoint keys:
 - epoch
 - model_state_dict
 - optimizer_state_dict
 - val_acc
 - val_loss
 - total_params
 - trainable_params


In [4]:
if isinstance(checkpoint, dict):
    print("\nTensor contents:")
    for key, value in checkpoint.items():
        if torch.is_tensor(value):
            print(key, value.shape)


Tensor contents:


In [5]:
state_dict = checkpoint["model_state_dict"]

print("Number of parameters:", len(state_dict))

for key, value in state_dict.items():
    print(f"{key:50s} {tuple(value.shape)}")

Number of parameters: 122
conv1.weight                                       (64, 1, 7, 7)
bn1.weight                                         (64,)
bn1.bias                                           (64,)
bn1.running_mean                                   (64,)
bn1.running_var                                    (64,)
bn1.num_batches_tracked                            ()
layer1.0.conv1.weight                              (64, 64, 3, 3)
layer1.0.bn1.weight                                (64,)
layer1.0.bn1.bias                                  (64,)
layer1.0.bn1.running_mean                          (64,)
layer1.0.bn1.running_var                           (64,)
layer1.0.bn1.num_batches_tracked                   ()
layer1.0.conv2.weight                              (64, 64, 3, 3)
layer1.0.bn2.weight                                (64,)
layer1.0.bn2.bias                                  (64,)
layer1.0.bn2.running_mean                          (64,)
layer1.0.bn2.running_var                  

In [6]:
import torch
import torch.nn as nn
from torchvision.models import resnet18

NUM_CLASSES = 16

model = resnet18(weights=None)

# Original project used 1-channel mel spectrograms
model.conv1 = nn.Conv2d(
    1,
    64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

# 16 music genres
model.fc = nn.Linear(512, NUM_CLASSES)

# Load trained checkpoint
model.load_state_dict(checkpoint["model_state_dict"])

model = model.cpu()
model.eval()

print(model)


ResNet(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [7]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

print("\nCheckpoint:")
print("Total parameters:", checkpoint["total_params"])
print("Trainable parameters:", checkpoint["trainable_params"])

Total parameters: 11178448
Trainable parameters: 11178448

Checkpoint:
Total parameters: 11178448
Trainable parameters: 8401936


In [8]:
print("Checkpoint epoch:", checkpoint["epoch"])
print("Checkpoint validation accuracy:", checkpoint["val_acc"])
print("Checkpoint validation loss:", checkpoint["val_loss"])

Checkpoint epoch: 6
Checkpoint validation accuracy: 0.6737220447284346
Checkpoint validation loss: 1.0804307624316825


In [10]:
TEST_DIR = PROJECT + "/data/features/test_final"

print("Test directory exists:", os.path.exists(TEST_DIR))

if os.path.exists(TEST_DIR):
    print("\nFiles:")
    for f in sorted(os.listdir(TEST_DIR)):
        print(" ", f)

Test directory exists: True

Files:
  checkpoints
  shard_000.npz
  shard_001.npz
  shard_002.npz
  shard_003.npz
  shard_004.npz
  shard_005.npz


In [11]:
import numpy as np

test_file = os.path.join(TEST_DIR, "shard_000.npz")

data = np.load(test_file)

print("Keys:", data.files)

for key in data.files:
    value = data[key]
    print(f"{key}: shape={value.shape}, dtype={value.dtype}")

Keys: ['track_id', 'genres', 'ann_features', 'mel_features']
track_id: shape=(500,), dtype=int64
genres: shape=(500,), dtype=<U19
ann_features: shape=(500, 80), dtype=float32
mel_features: shape=(500, 128, 1292), dtype=float16


In [12]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch

GENRE_NAMES = [
    "Blues",
    "Classical",
    "Country",
    "Easy Listening",
    "Electronic",
    "Experimental",
    "Folk",
    "Hip-Hop",
    "Instrumental",
    "International",
    "Jazz",
    "Old-Time / Historic",
    "Pop",
    "Rock",
    "Soul-RnB",
    "Spoken"
]

genre_to_idx = {
    genre: idx for idx, genre in enumerate(GENRE_NAMES)
}

class TestMelDataset(Dataset):
    def __init__(self, directory):
        self.samples = []

        shard_files = sorted([
            f for f in os.listdir(directory)
            if f.endswith(".npz") and f.startswith("shard_")
        ])

        print("Loading shards:", len(shard_files))

        for filename in shard_files:
            path = os.path.join(directory, filename)
            data = np.load(path)

            mel = data["mel_features"]
            genres = data["genres"]

            for i in range(len(mel)):
                self.samples.append(
                    (
                        mel[i],
                        genre_to_idx[genres[i]]
                    )
                )

        print("Total samples:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        mel, label = self.samples[index]

        # float16 → float32
        mel = torch.tensor(mel, dtype=torch.float32)

        # [-80, 0] → [0, 1]
        mel = (mel + 80.0) / 80.0

        # Add channel dimension
        mel = mel.unsqueeze(0)

        label = torch.tensor(label, dtype=torch.long)

        return mel, label


test_dataset = TestMelDataset(TEST_DIR)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Batches:", len(test_loader))

Loading shards: 6
Total samples: 2572
Batches: 81


In [13]:
x, y = next(iter(test_loader))

print("Input shape:", x.shape)
print("Input dtype:", x.dtype)
print("Input min:", x.min().item())
print("Input max:", x.max().item())
print("Labels shape:", y.shape)

Input shape: torch.Size([32, 1, 128, 1292])
Input dtype: torch.float32
Input min: 0.0
Input max: 1.0
Labels shape: torch.Size([32])


In [14]:
import time
import torch
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

model = model.cpu()
model.eval()

all_predictions = []
all_labels = []

# Warm-up
with torch.no_grad():
    warmup_x, _ = next(iter(test_loader))
    _ = model(warmup_x)

# Timed inference
start_time = time.perf_counter()

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.numpy())
        all_labels.extend(batch_y.numpy())

end_time = time.perf_counter()

total_time = end_time - start_time
num_samples = len(all_labels)

accuracy = accuracy_score(all_labels, all_predictions)

macro_precision = precision_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

latency_ms = (total_time / num_samples) * 1000

print("=" * 50)
print("FP32 RESNET18 CPU BASELINE")
print("=" * 50)
print(f"Test samples:       {num_samples}")
print(f"Accuracy:            {accuracy * 100:.4f}%")
print(f"Macro Precision:     {macro_precision:.4f}")
print(f"Macro Recall:        {macro_recall:.4f}")
print(f"Macro F1:            {macro_f1:.4f}")
print(f"Weighted F1:         {weighted_f1:.4f}")
print(f"Total inference:     {total_time:.4f} sec")
print(f"Latency/sample:      {latency_ms:.4f} ms")

KeyboardInterrupt: 

In [15]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [16]:
import time
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --------------------------------------------------
# 1. GPU SETUP
# --------------------------------------------------

device = torch.device("cuda")
print("Device:", torch.cuda.get_device_name(0))

model = model.to(device)
model.eval()

gpu_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Test samples:", len(test_dataset))
print("Batches:", len(gpu_loader))


# --------------------------------------------------
# 2. WARMUP
# --------------------------------------------------

print("\nWarming up GPU...")

with torch.inference_mode():
    for i, (x, y) in enumerate(gpu_loader):
        x = x.to(device, non_blocking=True)
        _ = model(x)

        if i >= 4:
            break

torch.cuda.synchronize()

print("Warmup complete.")


# --------------------------------------------------
# 3. FULL GPU INFERENCE
# --------------------------------------------------

predictions = []
actual_labels = []

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
    for x, y in gpu_loader:

        x = x.to(device, non_blocking=True)

        outputs = model(x)
        preds = torch.argmax(outputs, dim=1)

        predictions.append(preds)
        actual_labels.append(y.to(device))

torch.cuda.synchronize()
end_time = time.perf_counter()

elapsed = end_time - start_time


# --------------------------------------------------
# 4. RESULTS
# --------------------------------------------------

predictions = torch.cat(predictions).cpu().numpy()
actual_labels = torch.cat(actual_labels).cpu().numpy()

accuracy = accuracy_score(actual_labels, predictions)

macro_precision = precision_score(
    actual_labels,
    predictions,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    actual_labels,
    predictions,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    actual_labels,
    predictions,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    actual_labels,
    predictions,
    average="weighted",
    zero_division=0
)

latency_per_sample = (elapsed / len(test_dataset)) * 1000


print("\n" + "=" * 60)
print("FP32 RESNET18 — T4 GPU BASELINE")
print("=" * 60)

print(f"Accuracy:          {accuracy * 100:.4f}%")
print(f"Macro Precision:   {macro_precision:.4f}")
print(f"Macro Recall:      {macro_recall:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")

print(f"\nTotal inference:   {elapsed:.4f} sec")
print(f"Latency/sample:    {latency_per_sample:.4f} ms")

print(f"\nParameters:        {sum(p.numel() for p in model.parameters()):,}")
print("=" * 60)

Device: Tesla T4
Test samples: 2572
Batches: 41

Warming up GPU...
Warmup complete.

FP32 RESNET18 — T4 GPU BASELINE
Accuracy:          63.6858%
Macro Precision:   0.4517
Macro Recall:      0.3812
Macro F1:          0.3879
Weighted F1:       0.5974

Total inference:   9.4126 sec
Latency/sample:    3.6596 ms

Parameters:        11,178,448


In [17]:
import os
import torch

FP32_PATH = "/content/fp32_resnet18_state_dict.pth"

torch.save(model.state_dict(), FP32_PATH)

size_mb = os.path.getsize(FP32_PATH) / (1024 ** 2)

print("=" * 60)
print("FP32 MODEL SIZE")
print("=" * 60)
print(f"Model size: {size_mb:.2f} MB")
print(f"Parameters:  {sum(p.numel() for p in model.parameters()):,}")
print("=" * 60)

FP32 MODEL SIZE
Model size: 42.72 MB
Parameters:  11,178,448


In [18]:
import torch
import torch.nn as nn

print("Creating INT8 dynamic quantized model...")

# Start from the original FP32 model on CPU
fp32_cpu = model.cpu()
fp32_cpu.eval()

# Dynamic INT8 quantization
quantized_model = torch.ao.quantization.quantize_dynamic(
    fp32_cpu,
    {nn.Linear},
    dtype=torch.qint8
)

quantized_model.eval()

print("\nQuantized model created.")
print(quantized_model)

Creating INT8 dynamic quantized model...

Quantized model created.
ResNet(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=

/tmp/ipykernel_1545/3776372637.py:11: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.ao.quantization.quantize_dynamic(


In [19]:
import os
import torch

INT8_PATH = "/content/int8_resnet18_state_dict.pth"

torch.save(quantized_model.state_dict(), INT8_PATH)

size_mb = os.path.getsize(INT8_PATH) / (1024 ** 2)

print("=" * 60)
print("INT8 MODEL SIZE")
print("=" * 60)
print(f"FP32 size:  {42.72:.2f} MB")
print(f"INT8 size:  {size_mb:.2f} MB")
print(f"Reduction:  {(1 - size_mb / 42.72) * 100:.2f}%")
print("=" * 60)

INT8 MODEL SIZE
FP32 size:  42.72 MB
INT8 size:  42.70 MB
Reduction:  0.06%


In [20]:
import time
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from torch.utils.data import DataLoader

# --------------------------------------------------
# CPU SETUP
# --------------------------------------------------

quantized_model = quantized_model.cpu()
quantized_model.eval()

cpu_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Device: CPU")
print("Test samples:", len(test_dataset))
print("Batches:", len(cpu_loader))


# --------------------------------------------------
# WARMUP
# --------------------------------------------------

print("\nWarming up...")

with torch.inference_mode():
    for i, (x, y) in enumerate(cpu_loader):
        _ = quantized_model(x)

        if i >= 4:
            break

print("Warmup complete.")


# --------------------------------------------------
# FULL INFERENCE
# --------------------------------------------------

predictions = []
actual_labels = []

start_time = time.perf_counter()

with torch.inference_mode():
    for x, y in cpu_loader:

        outputs = quantized_model(x)
        preds = torch.argmax(outputs, dim=1)

        predictions.append(preds)
        actual_labels.append(y)

end_time = time.perf_counter()

elapsed = end_time - start_time


# --------------------------------------------------
# METRICS
# --------------------------------------------------

predictions = torch.cat(predictions).numpy()
actual_labels = torch.cat(actual_labels).numpy()

accuracy = accuracy_score(actual_labels, predictions)

macro_precision = precision_score(
    actual_labels,
    predictions,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    actual_labels,
    predictions,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    actual_labels,
    predictions,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    actual_labels,
    predictions,
    average="weighted",
    zero_division=0
)

latency_per_sample = (elapsed / len(test_dataset)) * 1000


# --------------------------------------------------
# RESULTS
# --------------------------------------------------

print("\n" + "=" * 60)
print("INT8 RESNET18 — CPU")
print("=" * 60)

print(f"Accuracy:          {accuracy * 100:.4f}%")
print(f"Macro Precision:   {macro_precision:.4f}")
print(f"Macro Recall:      {macro_recall:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")

print(f"\nTotal inference:   {elapsed:.4f} sec")
print(f"Latency/sample:    {latency_per_sample:.4f} ms")

print("=" * 60)

Device: CPU
Test samples: 2572
Batches: 81

Warming up...
Warmup complete.

INT8 RESNET18 — CPU
Accuracy:          63.8414%
Macro Precision:   0.4540
Macro Recall:      0.3832
Macro F1:          0.3907
Weighted F1:       0.6001

Total inference:   527.5675 sec
Latency/sample:    205.1196 ms


In [21]:
import torch
import torch.nn as nn
from torchvision.models import resnet18

# Recreate the original FP32 architecture
pruned_model = resnet18(weights=None)

pruned_model.conv1 = nn.Conv2d(
    1,
    64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

pruned_model.fc = nn.Linear(512, 16)

# Load the ORIGINAL best checkpoint
checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

pruned_model.load_state_dict(
    checkpoint["model_state_dict"]
)

pruned_model.eval()

print("Original FP32 model restored.")
print(
    "Parameters:",
    f"{sum(p.numel() for p in pruned_model.parameters()):,}"
)

Original FP32 model restored.
Parameters: 11,178,448


In [22]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

# --------------------------------------------------
# 30% GLOBAL UNSTRUCTURED PRUNING
# --------------------------------------------------

parameters_to_prune = []

for module in pruned_model.modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        parameters_to_prune.append((module, "weight"))

print("Layers being pruned:", len(parameters_to_prune))

prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.30
)

print("30% global pruning applied.")


# --------------------------------------------------
# CALCULATE ACTUAL SPARSITY
# --------------------------------------------------

total_weights = 0
zero_weights = 0

for module in pruned_model.modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        weight = module.weight.detach()

        total_weights += weight.numel()
        zero_weights += torch.sum(weight == 0).item()

sparsity = zero_weights / total_weights

print("\n" + "=" * 60)
print("PRUNING SUMMARY")
print("=" * 60)

print(f"Total prunable weights: {total_weights:,}")
print(f"Zero weights:           {zero_weights:,}")
print(f"Actual sparsity:        {sparsity * 100:.2f}%")
print("=" * 60)

Layers being pruned: 21
30% global pruning applied.

PRUNING SUMMARY
Total prunable weights: 11,168,832
Zero weights:           3,350,650
Actual sparsity:        30.00%


In [23]:
import time
import torch
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from torch.utils.data import DataLoader

device = torch.device("cuda")

pruned_model = pruned_model.to(device)
pruned_model.eval()

pruned_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Device:", torch.cuda.get_device_name(0))
print("Test samples:", len(test_dataset))
print("Batches:", len(pruned_loader))


# --------------------------------------------------
# WARMUP
# --------------------------------------------------

print("\nWarming up GPU...")

with torch.inference_mode():
    for i, (x, y) in enumerate(pruned_loader):
        x = x.to(device, non_blocking=True)
        _ = pruned_model(x)

        if i >= 4:
            break

torch.cuda.synchronize()

print("Warmup complete.")


# --------------------------------------------------
# INFERENCE
# --------------------------------------------------

predictions = []
actual_labels = []

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
    for x, y in pruned_loader:

        x = x.to(device, non_blocking=True)

        outputs = pruned_model(x)
        preds = torch.argmax(outputs, dim=1)

        predictions.append(preds)
        actual_labels.append(y.to(device))

torch.cuda.synchronize()
end_time = time.perf_counter()

elapsed = end_time - start_time


# --------------------------------------------------
# METRICS
# --------------------------------------------------

predictions = torch.cat(predictions).cpu().numpy()
actual_labels = torch.cat(actual_labels).cpu().numpy()

accuracy = accuracy_score(actual_labels, predictions)

macro_precision = precision_score(
    actual_labels,
    predictions,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    actual_labels,
    predictions,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    actual_labels,
    predictions,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    actual_labels,
    predictions,
    average="weighted",
    zero_division=0
)

latency_per_sample = (elapsed / len(test_dataset)) * 1000


# --------------------------------------------------
# RESULTS
# --------------------------------------------------

print("\n" + "=" * 60)
print("30% PRUNED RESNET18 — T4 GPU")
print("=" * 60)

print(f"Accuracy:          {accuracy * 100:.4f}%")
print(f"Macro Precision:   {macro_precision:.4f}")
print(f"Macro Recall:      {macro_recall:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")

print(f"\nTotal inference:   {elapsed:.4f} sec")
print(f"Latency/sample:    {latency_per_sample:.4f} ms")

print(f"\nSparsity:          {sparsity * 100:.2f}%")
print(f"Parameters:        {sum(p.numel() for p in pruned_model.parameters()):,}")

print("=" * 60)

Device: Tesla T4
Test samples: 2572
Batches: 41

Warming up GPU...
Warmup complete.

30% PRUNED RESNET18 — T4 GPU
Accuracy:          62.0140%
Macro Precision:   0.4615
Macro Recall:      0.3552
Macro F1:          0.3725
Weighted F1:       0.5717

Total inference:   9.7087 sec
Latency/sample:    3.7748 ms

Sparsity:          30.00%
Parameters:        11,178,448


In [24]:
import os
import torch

PRUNED_PATH = "/content/pruned_resnet18_state_dict.pth"

# Remove pruning reparameterization and permanently keep zero weights
for module in pruned_model.modules():
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
        if hasattr(module, "weight_orig"):
            torch.nn.utils.prune.remove(module, "weight")

pruned_model = pruned_model.cpu()

torch.save(pruned_model.state_dict(), PRUNED_PATH)

pruned_size_mb = os.path.getsize(PRUNED_PATH) / (1024 ** 2)

print("=" * 60)
print("PRUNED MODEL SIZE")
print("=" * 60)

print(f"FP32 size:     42.72 MB")
print(f"Pruned size:   {pruned_size_mb:.2f} MB")
print(f"Sparsity:      {sparsity * 100:.2f}%")
print(f"Parameters:    {sum(p.numel() for p in pruned_model.parameters()):,}")

print("=" * 60)

PRUNED MODEL SIZE
FP32 size:     42.72 MB
Pruned size:   42.72 MB
Sparsity:      30.00%
Parameters:    11,178,448


In [25]:
import os
import json
import torch

# ============================================================
# STAGE 6 — MODEL OPTIMIZATION ARTIFACTS
# ============================================================

PROJECT = "/content/drive/MyDrive/music-genre-capstone"

RESULTS_DIR = os.path.join(PROJECT, "results", "optimization")
MODELS_DIR = os.path.join(PROJECT, "models", "optimization")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("Results directory:", RESULTS_DIR)
print("Models directory:", MODELS_DIR)


# ============================================================
# 1. RECREATE CLEAN FP32 MODEL FROM BEST CHECKPOINT
# ============================================================

from torchvision.models import resnet18
import torch.nn as nn

fp32_model = resnet18(weights=None)

fp32_model.conv1 = nn.Conv2d(
    1,
    64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

fp32_model.fc = nn.Linear(512, 16)

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

fp32_model.load_state_dict(checkpoint["model_state_dict"])
fp32_model.eval()


# ============================================================
# 2. SAVE MODEL FILES
# ============================================================

FP32_MODEL_PATH = os.path.join(
    MODELS_DIR,
    "resnet18_fp32.pth"
)

INT8_MODEL_PATH = os.path.join(
    MODELS_DIR,
    "resnet18_int8_dynamic.pth"
)

PRUNED_MODEL_PATH = os.path.join(
    MODELS_DIR,
    "resnet18_pruned_30.pth"
)

torch.save(fp32_model.state_dict(), FP32_MODEL_PATH)
torch.save(quantized_model.state_dict(), INT8_MODEL_PATH)
torch.save(pruned_model.state_dict(), PRUNED_MODEL_PATH)

print("\nModels saved.")


# ============================================================
# 3. GET ACTUAL FILE SIZES
# ============================================================

def get_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)


fp32_size = get_size_mb(FP32_MODEL_PATH)
int8_size = get_size_mb(INT8_MODEL_PATH)
pruned_size = get_size_mb(PRUNED_MODEL_PATH)


# ============================================================
# 4. RESULT DICTIONARIES
# ============================================================

fp32_results = {
    "model": "ResNet18",
    "variant": "FP32",
    "hardware": "Tesla T4 GPU",
    "parameters": 11178448,
    "model_size_mb": round(fp32_size, 4),
    "accuracy": 0.636858,
    "accuracy_percent": 63.6858,
    "macro_precision": 0.4517,
    "macro_recall": 0.3812,
    "macro_f1": 0.3879,
    "weighted_f1": 0.5974,
    "inference_time_seconds": 9.4126,
    "latency_ms_per_sample": 3.6596,
    "sparsity_percent": 0.0
}

int8_results = {
    "model": "ResNet18",
    "variant": "Dynamic INT8",
    "hardware": "CPU",
    "parameters": 11178448,
    "model_size_mb": round(int8_size, 4),
    "accuracy": 0.638414,
    "accuracy_percent": 63.8414,
    "macro_precision": 0.4540,
    "macro_recall": 0.3832,
    "macro_f1": 0.3907,
    "weighted_f1": 0.6001,
    "inference_time_seconds": 527.5675,
    "latency_ms_per_sample": 205.1196,
    "sparsity_percent": None,
    "quantization_type": "Dynamic INT8",
    "quantized_layers": "Linear only"
}

pruned_results = {
    "model": "ResNet18",
    "variant": "30% Global Unstructured Pruning",
    "hardware": "Tesla T4 GPU",
    "parameters": 11178448,
    "model_size_mb": round(pruned_size, 4),
    "accuracy": 0.620140,
    "accuracy_percent": 62.0140,
    "macro_precision": 0.4615,
    "macro_recall": 0.3552,
    "macro_f1": 0.3725,
    "weighted_f1": 0.5717,
    "inference_time_seconds": 9.7087,
    "latency_ms_per_sample": 3.7748,
    "sparsity_percent": 30.0,
    "pruning_type": "Global L1 Unstructured"
}


# ============================================================
# 5. SAVE INDIVIDUAL JSON FILES
# ============================================================

with open(
    os.path.join(RESULTS_DIR, "fp32_results.json"),
    "w"
) as f:
    json.dump(fp32_results, f, indent=4)

with open(
    os.path.join(RESULTS_DIR, "int8_results.json"),
    "w"
) as f:
    json.dump(int8_results, f, indent=4)

with open(
    os.path.join(RESULTS_DIR, "pruning_30_results.json"),
    "w"
) as f:
    json.dump(pruned_results, f, indent=4)


# ============================================================
# 6. COMBINED COMPARISON FILE
# ============================================================

comparison = {
    "project": "Music Genre Classification using Deep Learning",
    "optimization_stage": "Stage 6 — Model Optimization",
    "base_model": "ResNet18",
    "test_samples": 2572,
    "results": {
        "fp32": fp32_results,
        "dynamic_int8": int8_results,
        "pruning_30_percent": pruned_results
    }
}

COMPARISON_PATH = os.path.join(
    RESULTS_DIR,
    "optimization_comparison.json"
)

with open(COMPARISON_PATH, "w") as f:
    json.dump(comparison, f, indent=4)


# ============================================================
# 7. FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 65)
print("STAGE 6 ARTIFACTS SAVED")
print("=" * 65)

print("\nRESULTS:")
for filename in sorted(os.listdir(RESULTS_DIR)):
    path = os.path.join(RESULTS_DIR, filename)
    print(f"  {filename:35s} {get_size_mb(path):.4f} MB")

print("\nMODELS:")
for filename in sorted(os.listdir(MODELS_DIR)):
    path = os.path.join(MODELS_DIR, filename)
    print(f"  {filename:35s} {get_size_mb(path):.2f} MB")

print("\n" + "=" * 65)
print("Stage 6 complete.")
print("=" * 65)

Results directory: /content/drive/MyDrive/music-genre-capstone/results/optimization
Models directory: /content/drive/MyDrive/music-genre-capstone/models/optimization

Models saved.

STAGE 6 ARTIFACTS SAVED

RESULTS:
  fp32_results.json                   0.0004 MB
  int8_results.json                   0.0005 MB
  optimization_comparison.json        0.0020 MB
  pruning_30_results.json             0.0005 MB

MODELS:
  resnet18_fp32.pth                   42.72 MB
  resnet18_int8_dynamic.pth           42.70 MB
  resnet18_pruned_30.pth              42.72 MB

Stage 6 complete.


In [3]:
import shutil

shutil.make_archive(
    "/content/music_genre_models",
    "zip",
    "/content/drive/MyDrive/music-genre-capstone/models"
)

print("Created /content/music_genre_models.zip")

Created /content/music_genre_models.zip


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from google.colab import files

files.download("/content/music_genre_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>